# Setting Up a Travis County Redistricting Analysis

**Scenario:** A new analyst joins and needs to set up a project directory
for a Travis County redistricting analysis. The setup must be:
- **Atomic** — config writes either fully succeed or leave the old version intact
- **Secure** — no path traversal, no accidental writes outside the project
- **Safe** — shell commands restricted to a known allow-list

siege_utilities provides all three guarantees through its files/ package.

## 1. Create the Project Structure

Every analysis project follows the same directory layout.
`ensure_directory_exists` creates nested paths idempotently — running it
twice produces the same result.

In [1]:
import tempfile
from pathlib import Path
from siege_utilities.files.operations import (
    ensure_directory_exists, safe_json_write, safe_json_read,
    safe_file_write, safe_file_read, file_exists,
)

# Work in a temp directory (would be a real project root in production)
project_root = Path(tempfile.mkdtemp()) / "travis_county_redistricting"

for subdir in ["data/raw", "data/processed", "output/maps", "output/reports", "logs"]:
    ensure_directory_exists(str(project_root / subdir))

# Idempotent — running again is a no-op
ensure_directory_exists(str(project_root / "data/raw"))

# Verify structure
for p in sorted(project_root.rglob("*")):
    if p.is_dir():
        rel = p.relative_to(project_root)
        print(f"  {rel}/")

  data/
  data/processed/
  data/raw/
  logs/
  output/
  output/maps/
  output/reports/


## 2. Atomic Configuration Writes

Project config must never be half-written. If the process crashes mid-write,
the old config must survive. `atomic_write_path` writes to a temp file first,
then atomically renames it — either the full new content lands or the old file stays.

In [2]:
import json
from siege_utilities.files.operations import atomic_write_path

config_path = project_root / "config.json"
config = {
    "project_name": "Travis County Redistricting Analysis",
    "state_fips": "48",
    "county_fips": "453",
    "census_vintage": 2020,
    "districts": ["TX-21", "TX-25", "TX-35"],
    "output_format": "geojson",
}

# Write config atomically
with atomic_write_path(config_path) as tmp:
    tmp.write_text(json.dumps(config, indent=2))

# Verify it landed correctly
loaded = safe_json_read(str(config_path))
print(f"Config written: {loaded['project_name']}")
print(f"Districts: {loaded['districts']}")

# Simulate a crash during update — original survives
try:
    with atomic_write_path(config_path) as tmp:
        tmp.write_text(json.dumps({"partial": True}))
        raise RuntimeError("simulated crash")
except RuntimeError:
    pass

restored = safe_json_read(str(config_path))
print(f"\nAfter crash, config intact: {restored['project_name']}")
assert restored == config, "Original config must survive a crash"

Config written: Travis County Redistricting Analysis
Districts: ['TX-21', 'TX-25', 'TX-35']

After crash, config intact: Travis County Redistricting Analysis


## 3. Path Security: Blocking Traversal Attacks

When processing user-supplied file paths (e.g., uploaded filenames),
path traversal attacks can escape the project directory. The validation
module blocks these attempts before any I/O happens.

In [3]:
from siege_utilities.files.validation import validate_safe_path, PathSecurityError

# Safe paths pass validation
validate_safe_path("data/raw/precincts.geojson")
print("Safe path accepted: data/raw/precincts.geojson")

# Traversal attempts are blocked
attack_paths = [
    "../../etc/passwd",
    "data/../../../secrets.env",
    "~/private_keys/id_rsa",
]

for path in attack_paths:
    try:
        validate_safe_path(path)
        print(f"  UNEXPECTED: {path} was accepted")
    except PathSecurityError as e:
        print(f"  Blocked: {path}")

Safe path accepted: data/raw/precincts.geojson
  Blocked: ../../etc/passwd
  Blocked: data/../../../secrets.env
  Blocked: ~/private_keys/id_rsa


## 4. Safe Shell Execution

When the analysis requires shell commands (e.g., `wc` to count lines in
a large CSV), `run_command` restricts execution to a known allow-list.
Commands not on the list raise immediately — no execution, no risk.

In [4]:
from siege_utilities.files.operations import run_command
from siege_utilities.files.shell import SecurityError

# Write sample data to count
sample_data = "precinct_id,dem_pct,gop_pct\n" + "".join(
    f"{i:04d},{50+i*0.1:.1f},{50-i*0.1:.1f}\n" for i in range(100)
)
data_file = str(project_root / "data/raw/precincts.csv")
safe_file_write(data_file, sample_data)

# Allowed command: wc -l
result = run_command(f"wc -l {data_file}")
print(f"Line count: {result.stdout.strip()}")

# Disallowed command: blocked before execution
try:
    run_command("python3 --version")
except SecurityError as e:
    print(f"\nBlocked: python3 — {e}")

Security validation failed: Command 'python3' not allowed. Allowed commands: ['cat', 'date', 'echo', 'find', 'grep', 'head', 'hostname', 'ls', 'pwd', 'tail', 'uname', 'wc', 'which', 'whoami']


Line count: 101 /var/folders/9h/83s_sxx17hv5gkccscgbyvzw0000gn/T/tmp8lg9m1_q/travis_county_redistricting/data/raw/precincts.csv

Blocked: python3 — Command 'python3' not allowed. Allowed commands: ['cat', 'date', 'echo', 'find', 'grep', 'head', 'hostname', 'ls', 'pwd', 'tail', 'uname', 'wc', 'which', 'whoami']


## Summary

A complete project setup with four safety guarantees:
- **Idempotent directories** — `ensure_directory_exists` is safe to call repeatedly
- **Atomic config writes** — `atomic_write_path` prevents half-written configs
- **Path validation** — `validate_safe_path` blocks traversal before any I/O
- **Command restriction** — `run_command` only executes allow-listed programs